In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score, cross_val_predict
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# STEP 2: Upload CSV dataset
uploaded = files.upload()
filename = next(iter(uploaded))

# STEP 3: Load dataset
data = pd.read_csv(filename)

print("FIRST FIVE ROWS")
print(data.head())

print("\nDATASET SHAPE")
print(data.shape)

print("\nDATASET INFORMATION")
data.info()

print("\nMISSING VALUES")
print(data.isnull().sum())

# STEP 4: Clean dataset
data = data.dropna().copy()

# Convert categorical columns into numeric columns if present
data = pd.get_dummies(data, drop_first=True, dtype=int)

# Check target column
if "Yield_ton" not in data.columns:
    raise ValueError("Dataset must contain a 'Yield_ton' column.")

if len(data) < 5:
    raise ValueError("At least 5 complete rows are required for 5-fold CV.")

# STEP 5: Separate input and target
X = data.drop("Yield_ton", axis=1)
y = data["Yield_ton"]

print("\nINPUT FEATURES")
print(X.head())

print("\nTARGET VARIABLE")
print(y.head())

# STEP 6: Create model and K-Fold
model = LinearRegression()
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# STEP 7: K-Fold Cross-Validation
scores = cross_val_score(model, X, y, cv=kf, scoring="r2")

print("\n5-FOLD CROSS-VALIDATION R2 SCORES")
for i, score in enumerate(scores, start=1):
    print(f"Fold {i}: {score:.4f}")

print("Average R2 Score:", round(scores.mean(), 4))

# STEP 8: Cross-validated predictions
predictions = cross_val_predict(model, X, y, cv=kf)
errors = y.to_numpy() - predictions

# STEP 9: Model evaluation
mse = mean_squared_error(y, predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y, predictions)
r2 = r2_score(y, predictions)

print("\nMODEL EVALUATION")
print("MSE:", round(mse, 4))
print("RMSE:", round(rmse, 4))
print("MAE:", round(mae, 4))
print("R2 Score:", round(r2, 4))

# GRAPH 1: K-Fold Cross-Validation Scores
plt.figure(figsize=(8, 5))
plt.bar(range(1, 6), scores)
plt.axhline(scores.mean(), linestyle="--",
            label=f"Average R2 = {scores.mean():.3f}")
plt.xlabel("Fold Number")
plt.ylabel("R2 Score")
plt.title("5-Fold Cross-Validation Scores")
plt.xticks(range(1, 6))
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.show()

# GRAPH 2: Actual vs Predicted Yield
plt.figure(figsize=(8, 5))
plt.scatter(y, predictions)
low = min(y.min(), predictions.min())
high = max(y.max(), predictions.max())
plt.plot([low, high], [low, high], "r--", label="Perfect prediction")
plt.xlabel("Actual Yield (ton)")
plt.ylabel("Predicted Yield (ton)")
plt.title("Actual vs Predicted Plant Yield")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# GRAPH 3: Prediction Error Plot
plt.figure(figsize=(8, 5))
plt.scatter(predictions, errors)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Yield (ton)")
plt.ylabel("Prediction Error (Actual - Predicted)")
plt.title("Prediction Error Plot")
plt.grid(alpha=0.3)
plt.show()

# GRAPH 4: Yield Distribution Histogram
plt.figure(figsize=(8, 5))
plt.hist(y, bins=10, edgecolor="black")
plt.xlabel("Yield (ton)")
plt.ylabel("Frequency")
plt.title("Distribution of Plant Yield")
plt.grid(axis="y", alpha=0.3)
plt.show()

# GRAPH 5: Correlation Heatmap
correlation = data.corr(numeric_only=True)

plt.figure(figsize=(10, 7))
sns.heatmap(correlation, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap of Plant Yield Dataset")
plt.show()

# Fit a model on the full dataset for coefficient visualization
model.fit(X, y)

# GRAPH 6: Linear Regression Coefficients
coefficients = pd.Series(model.coef_, index=X.columns).sort_values()

plt.figure(figsize=(9, 5))
coefficients.plot(kind="barh")
plt.xlabel("Coefficient Value")
plt.ylabel("Input Features")
plt.title("Linear Regression Feature Coefficients")
plt.grid(axis="x", alpha=0.3)
plt.show()

# GRAPH 7: Actual and Predicted Yield by Record
plt.figure(figsize=(10, 5))
record_numbers = np.arange(1, len(y) + 1)
plt.plot(record_numbers, y.to_numpy(), marker="o",
         label="Actual Yield")
plt.plot(record_numbers, predictions, marker="x",
         label="Predicted Yield")
plt.xlabel("Record Number")
plt.ylabel("Yield (ton)")
plt.title("Actual vs Predicted Yield by Record")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# GRAPH 8: Residual Distribution
plt.figure(figsize=(8, 5))
plt.hist(errors, bins=10, edgecolor="black")
plt.axvline(0, linestyle="--")
plt.xlabel("Prediction Error")
plt.ylabel("Frequency")
plt.title("Distribution of Prediction Errors")
plt.grid(axis="y", alpha=0.3)
plt.show()

# GRAPH 9: Feature Correlation with Yield
yield_correlation = correlation["Yield_ton"].drop("Yield_ton")
yield_correlation = yield_correlation.sort_values()

plt.figure(figsize=(9, 5))
yield_correlation.plot(kind="barh")
plt.axvline(0, linestyle="--")
plt.xlabel("Correlation with Yield")
plt.ylabel("Input Feature")
plt.title("Feature Correlation with Plant Yield")
plt.grid(axis="x", alpha=0.3)
plt.show()

# GRAPH 10: Box Plot of Yield
plt.figure(figsize=(8, 4))
sns.boxplot(x=y)
plt.xlabel("Yield (ton)")
plt.title("Box Plot of Plant Yield")
plt.grid(axis="x", alpha=0.3)
plt.show()

# GRAPH 11: Actual vs Predicted Regression Plot
plt.figure(figsize=(8, 5))
sns.regplot(x=y, y=predictions)
plt.xlabel("Actual Yield (ton)")
plt.ylabel("Predicted Yield (ton)")
plt.title("Regression Plot: Actual vs Predicted Yield")
plt.grid(alpha=0.3)
plt.show()

# STEP 10: Example prediction
# Demonstration using the first row's input features
new_data = X.iloc[[0]]
new_prediction = model.predict(new_data)

print("\nEXAMPLE PREDICTION")
print(new_data)
print("Predicted Yield (ton):", round(new_prediction[0], 2))

print("\nPROJECT COMPLETED SUCCESSFULLY!")
